# Step 0 - 파노라마 전처리 (6등분 중 2번째 구간 = 정면)

**목표**: 360° 파노라마(스트리트뷰 raw)를 수직으로 **6등분**하여 **2번째 구간(정면)**만 추출해
**Front view**를 생성한다. (좌·정면·우·뒤·아래·하늘 배열에서 인덱스 1이 정면)

**입력**: `img/2021/*.jpg` (360° 파노라마)

**출력**: `output/00_front/{stem}_front.jpg` (정면 크롭) + `{stem}_cam.json` (hfov, K, out_hw)

> 2번째 1/6 구간은 360° 중 60° 범위에 해당하므로 HFOV 60°로 핀홀 K를 계산한다(03 BEV에서 사용).


## 1. 라이브러리 및 파라미터

In [4]:
import json
from pathlib import Path

import cv2
import numpy as np
from tqdm.auto import tqdm

# 입출력 경로 ──────────────────────────────────────────────────────────
IMG_DIR = Path("img")
OUT_DIR = Path("output/00_front")
OUT_DIR.mkdir(parents=True, exist_ok=True)

# 정면 크롭 파라미터 ────────────────────────────────────────────────────
# 스트리트뷰 raw 파노라마: (좌, 정면, 우, 뒤, 아래, 하늘) 6구간
N_SPLIT   = 6    # 수직선으로 나눌 구간 수
FRONT_IDX = 1    # 0-based: 2번째 구간 = 정면

PANO_HFOV  = 360.0
FRONT_HFOV = PANO_HFOV / N_SPLIT   # = 60°

IMG_PATHS = sorted(IMG_DIR.glob("*.jpg"))
print(f"파노라마 {len(IMG_PATHS)}장 발견")


파노라마 36367장 발견


## 2. 6등분 중 2번째 구간 크롭 함수

파노라마를 수평으로 6등분하여 2번째 구간 `[w/6 : w/3]`을 잘라 Front view를 생성.

해당 구간은 360° 중 60° 범위이므로, 03 BEV의 핀홀 투영용 핀홀 intrinsics를 계산한다.
- `fx = (W/2) / tan(HFOV/2)`, `fy = fx`(정사각 픽셀 가정), `cx=W/2`, `cy=H/2`.


In [5]:
def make_front_crop(pano_bgr, n_split=N_SPLIT, front_idx=FRONT_IDX,
                    front_hfov=FRONT_HFOV):
    """파노라마를 수평 n등분 후 front_idx 구간 크롭. (크롭 이미지, 핀홀 K) 반환."""
    ph, pw = pano_bgr.shape[:2]
    seg_w = pw // n_split
    x0 = front_idx * seg_w
    x1 = x0 + seg_w
    front = pano_bgr[:, x0:x1].copy()
    fh, fw = front.shape[:2]

    # 핀홀 카메라 intrinsics (수평 화각 = front_hfov)
    fx = (fw / 2.0) / np.tan(np.radians(front_hfov) / 2.0)
    cx, cy = fw / 2.0, fh / 2.0
    K = np.array([[fx, 0, cx], [0, fx, cy], [0, 0, 1]], dtype=np.float64)
    return front, K


print("크롭 함수 정의 완료")


크롭 함수 정의 완료


## 3. 전체 이미지 처리 및 저장

In [6]:
for img_path in tqdm(IMG_PATHS, desc="전처리"):
    stem = img_path.stem
    pano = cv2.imread(str(img_path))
    ph, pw = pano.shape[:2]

    front, K = make_front_crop(pano)
    fh, fw = front.shape[:2]

    # 저장: 정면 크롭
    out_img = OUT_DIR / f"{stem}_front.jpg"
    cv2.imwrite(str(out_img), front)

    # 저장: 카메라 메타 (02·03이 공유)
    cam = {
        "src_image": img_path.name,
        "src_hw": [ph, pw],
        "out_hw": [fh, fw],
        "hfov_deg": FRONT_HFOV,
        "split": [N_SPLIT, FRONT_IDX],
        "K": K.tolist(),
        "fx": K[0, 0], "fy": K[1, 1], "cx": K[0, 2], "cy": K[1, 2],
    }
    with open(OUT_DIR / f"{stem}_cam.json", "w", encoding="utf-8") as f:
        json.dump(cam, f, ensure_ascii=False, indent=2)

print(f"=== 전처리 완료: {len(IMG_PATHS)}장 저장됨 → {OUT_DIR} ===")
print(f"다음 단계: 01_segmentation.ipynb")

전처리: 100%|██████████| 36367/36367 [01:37<00:00, 372.24it/s]

=== 전처리 완료: 36367장 저장됨 → output\00_front ===
다음 단계: 01_segmentation.ipynb
